## Import

In [1]:
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import optuna

from sklearn.metrics import root_mean_squared_error, mean_absolute_error, max_error

In [2]:
from ML.utils.utils import *
from ML.utils.Data_preparator import Data_preparator
from ML.utils.Model_evaluator import Model_evaluator
from ML.utils.Model_trainer import Model_trainer
from physics.Iso_data_handler import Iso_data_handler
from physics.Data_visualiser import Data_visualiser

In [ ]:
pre_path = "../../../../../../../"
physical_model = "PARSEC"
path_to_data = pre_path + "data/PARSEC/"
path_to_results = pre_path + "results/model_A/fine_tuned_models/"
path_to_predictions = pre_path + "predictions/model_A/fine_tuned_models/"
tag = "Base"
output_parameters = ['mass', 'radius']

## Data preparation

In [ ]:
iso_handler = Iso_data_handler(path_to_data, 
                              ['logAge', 'logTe', 'logg', 'label', 'metallicity', 'Mass', 'Rpol'], 
                              physical_model, reclassify=True)

iso_df = iso_handler.get_isochrone_dataframe()

Reading MIST dataframe from csv file...


In [ ]:
phase_filtered_iso_df = Data_preparator.filter_data(iso_df, {'label':[1, 2, 3, 4, 5, 6, 7, 8]})

In [ ]:
X_train, X_test, y_train, y_test = \
    Data_preparator.split_data(phase_filtered_iso_df, x_cols=['logAge', 'logTe', 'logg', 'metallicity'], 
                               y_cols=['Mass', 'Rpol'], random_state=12, print_stats=True)

Training set statistics:
Range in train data for the star_mass parameter : 0.0999979840073621 - 298.5447575808816
Median value in train data for the star_mass parameter: 2.0816081316727946
Mean value in train data for the star_mass parameter: 7.558407372495925

Range in train data for the log_R parameter : -0.9974747647513328 - 3.129269620812593
Median value in train data for the log_R parameter: 1.4993114860984695
Mean value in train data for the log_R parameter: 1.3944707591667809

Testing set statistics:
Range in test data for the star_mass parameter : 0.0999981896729906 - 296.5221171165397
Median value in test data for the star_mass parameter: 2.082595606409119
Mean value in test data for the star_mass parameter: 7.471864103970097

Range in test data for the log_R parameter : -0.9974234436680278 - 3.1297545143214007
Median value in test data for the log_R parameter: 1.5026448988619927
Mean value in test data for the log_R parameter: 1.396340263115711



## Fine-tuning

### XGBoost

In [40]:
def XGB_objective(trial, X_train, y_train, X_test, y_test, output_parameters):
    # Hyperparameters which will be tuned
    # n_estimators = trial.suggest_int("n_estimators", 75, 200)
    booster = trial.suggest_categorical("booster", ["gbtree", "dart"])
    max_depth = trial.suggest_int("max_depth", 3, 50)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    gamma = trial.suggest_int("gamma", 0, 10)
    learning_rate = trial.suggest_float("learning_rate", 0.1, 0.4)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    n_jobs = 10

    mdl = XGBRegressor(
        max_depth = max_depth,
        min_child_weight = min_child_weight,
        gamma = gamma,
        learning_rate = learning_rate,
        subsample = subsample,
        n_jobs = n_jobs
    )
    mdl.fit(X_train, y_train)
    preds = mdl.predict(X_test) # shape of preds = [[mass_1, radius_1], [mass_2, radius_2], ...]
    # truth, preds = Model_trainer.Kfold_pipeline(XGBRegressor, X_train_data=X_train, y_train_data=y_train, n_splits=10, 
    #                                             max_depth = max_depth,
    #                                             min_child_weight = min_child_weight,
    #                                             gamma = gamma,
    #                                             learning_rate = learning_rate,
    #                                             subsample = subsample,
    #                                             n_jobs = n_jobs)
    metrics_dict = dict()
    for i, output_param in enumerate(output_parameters):
        metrics_dict[output_param] = dict()
        metrics_dict[output_param]["RMSE"] = root_mean_squared_error(y_test[:, i], preds[:, i])
        metrics_dict[output_param]["MAE"] = mean_absolute_error(y_test[:, i], preds[:, i])
        metrics_dict[output_param]["MAX_ER"] = max_error(y_test[:, i], preds[:, i])


    return metrics_dict["mass"]["RMSE"], metrics_dict["mass"]["MAE"], metrics_dict["mass"]["MAX_ER"]

In [50]:
study_XGB = optuna.create_study(directions=["minimize", "minimize", "minimize"]) # we want to minimize the RMSE, MAE and MAX_ER
study_XGB.optimize(lambda trial: XGB_objective(trial, X_train, y_train, X_test, y_test, output_parameters), n_trials=100)

[I 2025-12-13 18:32:32,233] A new study created in memory with name: no-name-06914c0e-1675-45bd-8f37-d1db92587df6
[I 2025-12-13 18:32:34,292] Trial 0 finished with values: [1.408034828823188, 0.3313070968498888, 62.25462817126024] and parameters: {'booster': 'gbtree', 'max_depth': 7, 'min_child_weight': 6, 'gamma': 10, 'learning_rate': 0.29992661067594417, 'subsample': 0.9444939234338932}.
[I 2025-12-13 18:34:29,435] Trial 1 finished with values: [1.2925834986319789, 0.15710931249818785, 76.53653429918992] and parameters: {'booster': 'dart', 'max_depth': 40, 'min_child_weight': 1, 'gamma': 0, 'learning_rate': 0.14381581687823733, 'subsample': 0.8094640477052406}.
[I 2025-12-13 18:34:31,918] Trial 2 finished with values: [1.253507661310232, 0.21528033474120362, 65.49466418200242] and parameters: {'booster': 'dart', 'max_depth': 20, 'min_child_weight': 3, 'gamma': 9, 'learning_rate': 0.1978946388366176, 'subsample': 0.9680679870967843}.
[I 2025-12-13 18:34:35,574] Trial 3 finished with v

In [53]:
for trial in study_XGB.best_trials:
    print(f"[RMSE, MAE, MAX_ER] : {trial.values} \n parameters : {trial.params}")


[RMSE, MAE, MAX_ER] : [1.2075307660925019, 0.19751183794580499, 67.78043078356492] 
 parameters : {'booster': 'gbtree', 'max_depth': 25, 'min_child_weight': 7, 'gamma': 2, 'learning_rate': 0.1435826734412782, 'subsample': 0.7393308713706829}
[RMSE, MAE, MAX_ER] : [1.3491723464552219, 0.28029257286479475, 56.2999049147665] 
 parameters : {'booster': 'gbtree', 'max_depth': 8, 'min_child_weight': 5, 'gamma': 1, 'learning_rate': 0.17327940477484632, 'subsample': 0.6471969348327882}
[RMSE, MAE, MAX_ER] : [1.5220439241787902, 0.3308652698781761, 55.60794934836025] 
 parameters : {'booster': 'dart', 'max_depth': 7, 'min_child_weight': 2, 'gamma': 6, 'learning_rate': 0.1746871380029314, 'subsample': 0.7444829255502123}
[RMSE, MAE, MAX_ER] : [1.4154130751517884, 0.29115989759994226, 56.18054083697302] 
 parameters : {'booster': 'gbtree', 'max_depth': 8, 'min_child_weight': 5, 'gamma': 8, 'learning_rate': 0.13649078760909214, 'subsample': 0.5873198363881101}
[RMSE, MAE, MAX_ER] : [1.208292199791

In [1]:
best_params_XGB = 

xgb_evaluator = Model_evaluator("XGBoost", path=path_to_results, physical_model=physical_model, output_parameters=["mass", "radius"])
xgb_evaluator.evaluate_Kfold_results(XGBRegressor, X_train, y_train, path_to_predictions, tag, random_state=12, override=True, use_preds=False, **best_params_XGB, n_jobs=10)

SyntaxError: invalid syntax (1785730911.py, line 1)

### Random forest

In [ ]:
def RF_objective(trial, X_train, y_train, X_test, y_test):
    # Hyperparameters which will be tuned
    criterion = trial.suggest_categorical("criterion", ["squared_error", "friedman_mse", "poisson", "absolute_error"])
    # n_estimators = trial.suggest_int("n_estimators", 75, 125)
    max_depth = trial.suggest_int("max_depth", 3, 50)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)

    mdl = RandomForestRegressor(
        criterion = criterion,
        max_depth = max_depth,
        min_samples_split = min_samples_split,
        min_samples_leaf = min_samples_leaf,
        n_jobs = 10
    )
    mdl.fit(X_train, y_train)
    preds = mdl.predict(X_test)
    # truth, preds = Model_trainer.Kfold_pipeline(RandomForestRegressor, X_train_data=X_train, y_train_data=y_train, n_splits=10, 
    #                                             criterion = criterion,
    #                                             # n_estimators = n_estimators,
    #                                             max_depth = max_depth,
    #                                             min_samples_split = min_samples_split,
    #                                             min_samples_leaf = min_samples_leaf,
    #                                             n_jobs = 10
    #                                             )
    metrics_dict = dict()
    for i, output_param in enumerate(output_parameters):
        metrics_dict[output_param] = dict()
        metrics_dict[output_param]["RMSE"] = root_mean_squared_error(y_test[:, i], preds[:, i])
        metrics_dict[output_param]["MAE"] = mean_absolute_error(y_test[:, i], preds[:, i])
        metrics_dict[output_param]["MAX_ER"] = max_error(y_test[:, i], preds[:, i])

    return metrics_dict["mass"]["RMSE"], metrics_dict["mass"]["MAE"], metrics_dict["mass"]["MAX_ER"]

In [ ]:
study_RF = optuna.create_study(directions=["minimize", "minimize", "minimize"]) # we want to minimize the RMSE, MAE and MAX_ER
study_RF.optimize(lambda trial: RF_objective(trial, X_train, y_train, X_test, y_test, output_parameters), n_trials=100)

In [ ]:
for trial in study_RF.best_trials:
    print(f"[RMSE, MAE, MAX_ER] : {trial.values} \n parameters : {trial.params}")

In [ ]:

best_params_RF = 

rf_evaluator = Model_evaluator("radnom_forest", path=path_to_results, physical_model=physical_model, output_parameters=["mass", "radius"])
rf_evaluator.evaluate_Kfold_results(RandomForestRegressor, X_train, y_train, path_to_predictions, tag, random_state=12, override=True, use_preds=False, **best_params_RF, n_jobs=10)